# WESAD Dataset - Exploratory Data Analysis
## Wearable Stress and Affect Detection

This notebook performs EDA on the preprocessed WESAD dataset (Subject S2).

**Signals:**
- ACC (3-axis accelerometer)
- ECG (electrocardiogram)
- EDA (electrodermal activity)
- EMG (Electromyography)
- Temp (temperature)
- Resp (respiration)

**Labels:**
- 1 = Baseline
- 2 = Stress
- 3 = Amusement

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt

# Path file from Kaggle and convert it to a dictionary format

file_path = 'C:/Downloads/S2.pkl' 

print(f"processing file: {file_path} ...")

with open(file_path, 'rb') as file:
    data = pickle.load(file, encoding='latin1')

print("✅ Data is loaded!")
print("Keys are:", data.keys())

Sedang membaca file: C:/Downloads/S2.pkl ...


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Downloads/S2.pkl'

In [19]:
data['signal']['chest'].keys()

dict_keys(['ACC', 'ECG', 'EMG', 'EDA', 'Temp', 'Resp'])

In [35]:

import os
import numpy as np
import pandas as pd

# High-quality downsampling for a non-integer ratio (700 -> 250 = 5/14).
try:
    from scipy.signal import resample_poly
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

# Parquet engine detection
PARQUET_ENGINE = None
try:
    import pyarrow  # noqa: F401
    PARQUET_ENGINE = "pyarrow"
except Exception:
    try:
        import fastparquet  # noqa: F401
        PARQUET_ENGINE = "fastparquet"
    except Exception:
        PARQUET_ENGINE = None

# ----- USER INPUTS -----
FS_ORIG = 700.0                          # original sampling frequency (Hz)
FS_TARGET = 250.0                        # target sampling frequency (Hz)
SIGNALS_PATH = ["signal", "chest"]       # adjust if needed (e.g., ["signal", "wrist"])
PARQUET_PATH = "chest_resampled_250Hz_with_label.parquet"
# ------------------------

# Known axis names for multi-channel sensors
AXIS_NAMES = {
    "ACC": ["ACC_x", "ACC_y", "ACC_z"]
}

def get_nested(d, path):
    cur = d
    for key in path:
        if key not in cur:
            raise KeyError(f"Missing key '{key}' at path {path}. Available here: {list(cur.keys())}")
        cur = cur[key]
    return cur

def ensure_2d_numeric(arr):
    a = np.asarray(arr)
    if a.ndim == 1:
        a = a[:, None]
    if a.ndim != 2:
        raise ValueError(f"Expected 1D/2D numeric array, got shape {a.shape}")
    if not np.issubdtype(a.dtype, np.number):
        raise ValueError(f"Array dtype must be numeric, got {a.dtype}")
    return a

def downsample_700_to_250_polyphase(x):
    if not SCIPY_AVAILABLE:
        raise RuntimeError(
            "SciPy is required for high-quality 700->250 Hz downsampling (resample_poly). "
            "Install: pip install scipy"
        )
    # 250/700 = 5/14
    return resample_poly(np.asarray(x), up=5, down=14, axis=0)

def map_labels_to_250Hz(labels, n_orig, n_out):
    """
    Map labels from original sample count (n_orig @ 700 Hz) to n_out @ 250 Hz.
    - If labels length == n_orig: nearest-index mapping.
    - If length == n_out: use as-is.
    - If scalar or length == 1: broadcast.
    - Otherwise: raise for clarity (to avoid guessing semantics).
    Returns a 1D array of length n_out.
    """
    lbl = np.asarray(labels)
    # Scalar
    if lbl.ndim == 0:
        return np.repeat(lbl.item(), n_out)
    # 1D vector of labels
    if lbl.ndim == 1:
        L = lbl.shape[0]
        if L == n_out:
            return lbl
        if L == n_orig:
            idx_map = np.linspace(0, n_orig - 1, n_out).astype(int)  # nearest
            return lbl[idx_map]
        if L == 1:
            return np.repeat(lbl[0], n_out)
        raise ValueError(
            f"Label length {L} does not match original ({n_orig}) or output ({n_out}) length. "
            "Please clarify label semantics."
        )
    # 2D (multi-label per sample) -> handle per column as above only if matches
    if lbl.ndim == 2:
        L = lbl.shape[0]
        if L == n_out:
            return lbl
        if L == n_orig:
            idx_map = np.linspace(0, n_orig - 1, n_out).astype(int)
            return lbl[idx_map, :]
        if L == 1:
            return np.repeat(lbl, n_out, axis=0)
        raise ValueError(
            f"Label rows {L} do not match original ({n_orig}) or output ({n_out})."
        )
    raise ValueError("Unsupported label array shape.")

def build_df_no_time(mods_resampled, labels_resampled=None):
    """
    Combine resampled modalities into one DataFrame (no time column).
    Truncate to shortest length to keep alignment. Add 'label' if provided.
    """
    lengths = [arr.shape[0] for arr in mods_resampled.values()]
    if labels_resampled is not None:
        lengths.append(labels_resampled.shape[0] if labels_resampled.ndim > 1 else labels_resampled.shape[0])
    n = min(lengths)

    df = pd.DataFrame(index=pd.RangeIndex(start=0, stop=n, step=1))
    df.index.name = "sample_idx"

    for name, arr in mods_resampled.items():
        arr = arr[:n, :]
        # Column naming
        if name in AXIS_NAMES and len(AXIS_NAMES[name]) == arr.shape[1]:
            cols = AXIS_NAMES[name]
        else:
            cols = [name] if arr.shape[1] == 1 else [f"{name}_{i+1}" for i in range(arr.shape[1])]
        for i, col_name in enumerate(cols):
            df[col_name] = arr[:, i].astype(np.float32)  # save space

    if labels_resampled is not None:
        labels_resampled = labels_resampled[:n]
        # 1D labels
        if labels_resampled.ndim == 1:
            df["label"] = labels_resampled
        else:
            for i in range(labels_resampled.shape[1]):
                df[f"label_{i+1}"] = labels_resampled[:, i]

    return df

def save_parquet(df, path, engine=PARQUET_ENGINE):
    if engine is None:
        raise RuntimeError(
            "No Parquet engine found (pyarrow or fastparquet). "
            "Install one:\n  pip install pyarrow\n  -- or --\n  pip install fastparquet"
        )
    df.to_parquet(path, index=True, engine=engine)
    print(f"Parquet saved: {os.path.abspath(path)} (rows={len(df)}, engine={engine})")

def main(data):
    # 1) Extract signals
    chest = get_nested(data, SIGNALS_PATH)

    # 2) Build modality dict (numeric arrays only)
    modalities = {}
    for name, arr in chest.items():
        try:
            a2d = ensure_2d_numeric(arr)
            modalities[name] = a2d
        except Exception:
            continue
    if not modalities:
        raise ValueError("No numeric modalities found under the specified path.")

    # 3) Downsample each modality from 700 Hz to 250 Hz
    mods_resampled = {}
    for name, arr in modalities.items():
        y = downsample_700_to_250_polyphase(arr)
        mods_resampled[name] = y
        print(f"{name}: {arr.shape} @ 700 Hz -> {y.shape} @ 250 Hz (polyphase)")

    # 4) Handle labels at data['label']
    labels_resampled = None
    if "label" in data:
        # Determine original N and downsampled N based on a reference modality
        # (use the shortest modality to avoid over-indexing)
        n_orig = min(a.shape[0] for a in modalities.values())
        n_out = min(b.shape[0] for b in mods_resampled.values())
        labels_resampled = map_labels_to_250Hz(data["label"], n_orig, n_out)

    # 5) Combine into DataFrame (no time column) and save Parquet
    out_df = build_df_no_time(mods_resampled, labels_resampled)
    save_parquet(out_df, PARQUET_PATH)

# ---- Run after loading your dict to `data` ----
main(data)


ACC: (4255300, 3) @ 700 Hz -> (1519750, 3) @ 250 Hz (polyphase)
ECG: (4255300, 1) @ 700 Hz -> (1519750, 1) @ 250 Hz (polyphase)
EMG: (4255300, 1) @ 700 Hz -> (1519750, 1) @ 250 Hz (polyphase)
EDA: (4255300, 1) @ 700 Hz -> (1519750, 1) @ 250 Hz (polyphase)
Temp: (4255300, 1) @ 700 Hz -> (1519750, 1) @ 250 Hz (polyphase)
Resp: (4255300, 1) @ 700 Hz -> (1519750, 1) @ 250 Hz (polyphase)
Parquet saved: c:\Users\Behrooz\Downloads\chest_resampled_250Hz_with_label.parquet (rows=1519750, engine=pyarrow)


In [36]:
# opening the parquet file

import pandas as pd

# Path to your Parquet file
parquet_path = "chest_resampled_250Hz_with_label.parquet"

# Read the Parquet file into a DataFrame
df = pd.read_parquet(parquet_path)

# Check the shape and first few rows
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(df.head())


Rows: 1519750, Columns: 9
               ACC_x     ACC_y     ACC_z       ECG       EMG       EDA  \
sample_idx                                                               
0           0.639947 -0.151158 -0.379472  0.014190  0.002472  3.564895   
1           0.972108 -0.230630 -0.562307  0.017409 -0.001401  5.664967   
2           0.842244 -0.180902 -0.404686  0.003917 -0.009106  5.055126   
3           0.868342 -0.182145 -0.366917  0.009662 -0.004851  5.376473   
4           0.787426 -0.164855 -0.316112  0.002525 -0.014659  5.175595   

                 Temp      Resp  label  
sample_idx                              
0           20.450800 -0.770499      0  
1           32.450565 -1.244406      0  
2           28.988943 -1.104927      0  
3           30.868170 -1.196511      0  
4           29.663376 -1.155388      0  
